# 01 — Setup: Dravnieks 1985 → Embeddings → Qdrant


```bash
pip install pyrfume sentence-transformers qdrant-client pandas numpy
```

## 1. Descargar Dravnieks 1985 via Pyrfume

In [3]:
import pyrfume
import pandas as pd
import numpy as np

molecules = pyrfume.load_data("dravnieks_1985/molecules.csv")
stimuli   = pyrfume.load_data("dravnieks_1985/stimuli.csv")
behavior  = pyrfume.load_data("dravnieks_1985/behavior_1.csv")

print("Moléculas:", molecules.shape)
print("Estímulos:", stimuli.shape)
print("Behavior (filas=moléculas, columnas=descriptores):", behavior.shape)

Moléculas: (138, 4)
Estímulos: (160, 4)
Behavior (filas=moléculas, columnas=descriptores): (160, 146)


In [4]:
behavior.head(3)

,"FRUITY,CITRUS",LEMON,GRAPEFRUIT,ORANGE,"FRUITY,OTHER THAN CITRUS",PINEAPPLE,GRAPE JUICE,STRAWBERRY,"APPLE, FRUIT",PEAR,...,"PUTRID, FOUL, DECAYED","FECAL, LIKE MANURE",CADAVEROUS,SICKENING,"DRY, POWDERY",CHALKY,LIGHT,HEAVY,"COOL,COOLING",WARM
Stimulus,,,,,,,,,,,,,,,,,,,,,
Abhexone_high,0.00,0.00,0.00,0.00,0.66,0.00,0.33,0.00,0.47,0.00,...,1.87,0.33,0.47,7.44,4.19,0.00,13.83,17.58,2.58,8.17
Acetophenone_high,1.57,0.52,0.26,0.41,8.29,0.52,1.26,1.74,0.00,1.02,...,1.63,0.58,0.68,10.14,4.08,0.00,12.06,29.34,9.89,17.95
AcetylPyridine_high,0.00,0.32,0.00,0.00,0.45,0.00,0.00,1.24,0.00,0.00,...,6.26,1.01,1.46,13.18,11.31,2.58,4.18,23.51,0.00,13.79


In [5]:
descriptors = list(behavior.columns)
print(f"Total descriptores: {len(descriptors)}")
print("Primeros 15:", descriptors[:15])

Total descriptores: 146
Primeros 15: ['FRUITY,CITRUS', 'LEMON', 'GRAPEFRUIT', 'ORANGE', 'FRUITY,OTHER THAN CITRUS', 'PINEAPPLE', 'GRAPE JUICE', 'STRAWBERRY', 'APPLE, FRUIT', 'PEAR', 'CANTALOUPE, HONEYDEW MELON', 'PEACH FRUIT', 'BANANA', 'FLORAL', 'ROSE']


In [6]:
print("Columnas de stimuli:", stimuli.columns.tolist())
stimuli.head(5)

Columnas de stimuli: ['CAS', 'CID', 'Conc', 'Name']


,CAS,CID,Conc,Name
Stimulus,,,,
Abhexone_high,698-10-2,61199.0,high,Abhexone
Acetophenone_high,98-86-2,7410.0,high,Acetophenone
AcetylPyridine_high,1122-62-9,14286.0,high,AcetylPyridine
Adoxal_high,141-13-9,98403.0,high,Adoxal
AldehydeC-16highcon_high,77-83-8,6501.0,high,AldehydeC-16highcon


In [7]:
name_candidates = [c for c in stimuli.columns 
                   if any(k in c.lower() for k in ['name', 'label', 'odor', 'compound', 'substance'])]
print("Columna de nombre detectada:", name_candidates)

mol_name_col = name_candidates[0] if name_candidates else None

behavior_named = behavior.copy()
if mol_name_col:
    behavior_named.insert(0, 'molecule_name', stimuli[mol_name_col].values)
else:
    behavior_named.insert(0, 'molecule_name', [f"molecule_{i}" for i in behavior.index])

print("\nEjemplo con nombres:")
behavior_named[['molecule_name'] + descriptors[:4]].head(5)

Columna de nombre detectada: ['Name']

Ejemplo con nombres:


,molecule_name,"FRUITY,CITRUS",LEMON,GRAPEFRUIT,ORANGE
Stimulus,,,,,
Abhexone_high,Abhexone,0.00,0.00,0.00,0.00
Acetophenone_high,Acetophenone,1.57,0.52,0.26,0.41
AcetylPyridine_high,AcetylPyridine,0.00,0.32,0.00,0.00
Adoxal_high,Adoxal,5.00,3.66,1.09,3.95
AldehydeC-16highcon_high,AldehydeC-16highcon,10.15,3.39,0.00,6.39


In [8]:
mol_ejemplo = behavior_named.iloc[0]
nombre = mol_ejemplo['molecule_name']
perfil = mol_ejemplo[descriptors]
top_descs = perfil[perfil > 20].sort_values(ascending=False)

print(f"Perfil olfativo de '{nombre}' (descriptores con score > 20):")
for desc, score in top_descs.items():
    print(f"  {desc:30s}  {score:.1f}%")

Perfil olfativo de 'Abhexone' (descriptores con score > 20):
  MAPLE SYRUP                     22.3%
  SWEET                           20.7%


In [9]:
behavior_named.to_csv("dravnieks_behavior_named.csv")
pd.Series(descriptors).to_csv("descriptor_list.csv", index=False, header=["descriptor"])
print("Archivos guardados: dravnieks_behavior_named.csv, descriptor_list.csv")

Archivos guardados: dravnieks_behavior_named.csv, descriptor_list.csv


## 2. Embedear los 146 descriptores

**Modelo:** `paraphrase-multilingual-MiniLM-L12-v2`


In [10]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
print(f"Cargando modelo {MODEL_NAME}...")
embed_model = SentenceTransformer(MODEL_NAME)
print("Modelo listo.")

Cargando modelo paraphrase-multilingual-MiniLM-L12-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4302.16it/s]


Modelo listo.


In [ ]:
print(f"Generando embeddings para {len(descriptors)} descriptores...")
descriptor_embeddings = embed_model.encode(
    descriptors,
    normalize_embeddings=True,   
    show_progress_bar=True
)
print(f"Shape: {descriptor_embeddings.shape}")  # esperado: (146, 384)

Generando embeddings para 146 descriptores...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [00:01<00:00,  4.40it/s]

Shape: (146, 384)


In [12]:
np.save("descriptor_embeddings.npy", descriptor_embeddings)
print("Guardado: descriptor_embeddings.npy")

Guardado: descriptor_embeddings.npy


## 3. Indexar en Qdrant

146 puntos en la colección, uno por descriptor.
El payload de cada punto guarda el nombre del descriptor para recuperarlo en la búsqueda.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant = QdrantClient(":memory:")

COLLECTION = "dravnieks_descriptors"
VECTOR_SIZE = descriptor_embeddings.shape[1]  # 384

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE)
)

points = [
    PointStruct(
        id=i,
        vector=descriptor_embeddings[i].tolist(),
        payload={"descriptor": descriptors[i]}
    )
    for i in range(len(descriptors))
]

qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"{len(points)} descriptores indexados.")

146 descriptores indexados.


In [ ]:
def test_busqueda(query, n=6):
    vec = embed_model.encode([query], normalize_embeddings=True)[0]
    hits = qdrant.query_points(collection_name=COLLECTION, query=vec.tolist(), limit=n).points
    print(f"\nTop {n} descriptores para '{query}':")
    for h in hits:
        print(f"  {h.payload['descriptor']:30s}  coseno={h.score:.4f}")

test_busqueda("warm wood")
test_busqueda("calidez acogedora")    
test_busqueda("fresh ocean breeze")


Top 6 descriptores para 'warm wood':
  WARM                            coseno=0.5463
  OAK WOOD,COGNAC                 coseno=0.4860
  STRAWBERRY                      coseno=0.4859
  BURNT CANDLE                    coseno=0.4694
  BUTTERY, FRESH BUTTER           coseno=0.4659
  BURNT PAPER                     coseno=0.4508

Top 6 descriptores para 'calidez acogedora':
  WARM                            coseno=0.7105
  FLORAL                          coseno=0.6740
  LIGHT                           coseno=0.6205
  MOLASSES                        coseno=0.6075
  BURNT CANDLE                    coseno=0.6016
  STRAWBERRY                      coseno=0.5739

Top 6 descriptores para 'fresh ocean breeze':
  LEATHER                         coseno=0.3935
  WARM                            coseno=0.3921
  FISHY                           coseno=0.3864
  WET PAPER                       coseno=0.3729
  FLORAL                          coseno=0.3340
  SEASONING FOR MEAT              coseno=0.3081


## Archivos generados:

| Archivo | Contenido |
|---|---|
| `descriptor_embeddings.npy` | Embeddings de los 146 descriptores (shape: 146×384) |
| `descriptor_list.csv` | Lista de descriptores en el mismo orden |
| `dravnieks_behavior_named.csv` | Matriz molécula × descriptor (scores 0-100) con nombres |

